# Fraud Detection with Blind Insight

This notebook demonstrates fraud detection using encrypted aggregations from Blind Insight.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the Blind Insight proxy is running

3. Upload the fraud dataset to Blind Insight

## Setup and Imports

In [ ]:
import numpy as np
import warnings

# Suppress SSL warnings for local development
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient

## Configuration

In [ ]:
# Blind Insight connection settings
ORGANIZATION = "demo"
DATASET_SLUG = "fraud-analysis-training"
SCHEMA_SLUG = "fraud-analysis-schema"
API_URL = "https://proxy.local.blindinsight.io/"

USERNAME = "data_owner@localhost"
PASSWORD = "blindinsight"

SCHEMA_ID = "iEbXomadeyp3JcDfZMFLYp"

# Features to analyze
FEATURES = ["amount", "device_risk_score", "ip_risk_score", "hour"]

# Schema max values for aggregation ranges
SCHEMA = {
    "amount": 12000,
    "device_risk_score": 100,
    "ip_risk_score": 100,
    "hour": 23,
    "dataset_order": 500
}

# Initialize client
client = BlindInsightClient(
    api_url=API_URL,
    username=USERNAME,
    password=PASSWORD,
    verify_ssl=False
)

print("Client initialized")

## Helper Function

In [ ]:
def agg_value(resp):
    """Extract aggregation value from Blind Insight response."""
    recs = resp.get("records", [])
    if not recs:
        return None
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else None
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else None
    return None

## Get Total Record Count (Encrypted)

In [ ]:
# Get total count using encrypted aggregation
count_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter="dataset_order:count(0~100000)",
    decrypt=False,
    schema_id=SCHEMA_ID
)
total_count = int(agg_value(count_resp))
print(f"Total records: {total_count}")

## Compute Global Means (Encrypted)

In [ ]:
# Compute mean for each feature across all records
print("Computing mean for each feature (encrypted):\n")

for feature in FEATURES:
    max_range = SCHEMA.get(feature, 10000)
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:avg(0~{max_range})",
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    mean_val = agg_value(resp)
    print(f"  mean {feature}: {mean_val:.3f}")

## Batch Aggregations Using dataset_order (Encrypted)

Compute means for subsets of the data using `dataset_order` to batch records.
This demonstrates how to perform aggregations on specific ranges without decrypting individual records.

In [ ]:
# Batch aggregation using dataset_order
batch_size = 50
num_batches = 5  # Just show first 5 batches

print(f"Computing batch means (batch_size={batch_size}):\n")

for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = start + batch_size - 1
    batch_filter = f"dataset_order:{start}~{end}"
    
    print(f"Batch {batch_idx} (records {start}-{end}):")
    
    for feature in FEATURES:
        max_range = SCHEMA.get(feature, 10000)
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~{max_range})",
            extra_filters=[batch_filter],
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        if mean_val is not None:
            print(f"    {feature}: {mean_val:.3f}")
    print()

## Means by Fraud Status (Encrypted)

Compare feature means between fraud and legitimate transactions.

In [ ]:
# Compute means by fraud status
print("Mean values by fraud status (encrypted):\n")
print(f"{'Feature':<25} {'Fraud':>12} {'Legitimate':>12} {'Difference':>12}")
print("-" * 65)

for feature in FEATURES:
    max_range = SCHEMA.get(feature, 10000)
    
    # Mean for fraud (is_fraud=1)
    resp_fraud = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:avg(0~{max_range})",
        extra_filters=["is_fraud:1~1"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    mean_fraud = agg_value(resp_fraud) or 0
    
    # Mean for legitimate (is_fraud=0)
    resp_legit = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:avg(0~{max_range})",
        extra_filters=["is_fraud:0~0"],
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    mean_legit = agg_value(resp_legit) or 0
    
    diff = mean_fraud - mean_legit
    print(f"{feature:<25} {mean_fraud:>12.3f} {mean_legit:>12.3f} {diff:>+12.3f}")

## Summary

This notebook demonstrated:

1. **Encrypted counts** - Getting total record count without decryption
2. **Encrypted means** - Computing feature averages on encrypted data
3. **Batch aggregations** - Using `dataset_order` to compute statistics on subsets
4. **Conditional aggregations** - Computing means by fraud status using `extra_filters`

All operations were performed on encrypted data - no individual records were decrypted.